# Final Project

本 Notebook 对应任务书第二阶段（期末报告 20%）的工程实现与运行指南。

## 核心职责分工

- **`Final_Project.py`** — 生成层：加载数据 → 构建推荐工件 → 生成 `outputs/submission.csv`
- **`Final_Project_Eval.py`** — 评估层：离线 5 折评估 + 规则权重调参
- **`Final_Project_LGBM.py`** — 增强版：LightGBM 排序 + 多源候选扩展（推荐在 Kaggle 运行）
- **`LGBM_Ablation_SHAP_Analysis.*`** — 可解释性：特征消融 + SHAP 分析

## Notebook 结构

| 章节 | 内容 |
|------|------|
| 1-3  | 环境检查、数据加载、生成层分步执行 |
| 4-5  | 一键生成和输出验证 |
| 6    | 评估层说明（耗时长，可选） |
| 7    | LightGBM 增强版（Kaggle 推荐） |
| 8    | 消融实验与可解释性分析 |
| 9    | 实验追踪与管理规范 |
| 10   | Kaggle 提交流程 |
| 11   | 数据路径配置 |
| 12-13| 故障排查与架构参考 |
| 14   | 进阶配置与场景模板 |
| 15   | 文档导航与文件验证 |

## 1. 导入依赖与环境检查

In [1]:
from pathlib import Path

from Final_Project import (
    BASE_PATH,
    OUTPUT_DIR,
    print_environment_info,
    print_data_file_status,
    load_tables,
    summarize_inputs,
    prepare_transactions,
    prepare_article_department,
    prepare_customer_age_bin,
    load_ranker_from_cache,
    describe_ranker,
    generate_submission,
    run_pipeline,
)

print_environment_info()
print_data_file_status(BASE_PATH)


Python: 3.12.1
Platform: Linux-6.8.0-1044-azure-x86_64-with-glibc2.39
polars: 1.40.1
articles     exists=False path=/workspaces/bigdata-HM/data/h-and-m-personalized-fashion-recommendations/articles.csv
customers    exists=False path=/workspaces/bigdata-HM/data/h-and-m-personalized-fashion-recommendations/customers.csv
transactions exists=False path=/workspaces/bigdata-HM/data/h-and-m-personalized-fashion-recommendations/transactions_train.csv
submission   exists=False path=/workspaces/bigdata-HM/data/h-and-m-personalized-fashion-recommendations/sample_submission.csv


{'articles': PosixPath('/workspaces/bigdata-HM/data/h-and-m-personalized-fashion-recommendations/articles.csv'),
 'customers': PosixPath('/workspaces/bigdata-HM/data/h-and-m-personalized-fashion-recommendations/customers.csv'),
 'transactions': PosixPath('/workspaces/bigdata-HM/data/h-and-m-personalized-fashion-recommendations/transactions_train.csv'),
 'submission': PosixPath('/workspaces/bigdata-HM/data/h-and-m-personalized-fashion-recommendations/sample_submission.csv')}

## 2. 查看输入数据概览

In [ ]:
tables = load_tables(BASE_PATH)
summary = summarize_inputs(tables)
summary

## 3. 生成层：仅产出 submission.csv

这一步是“手动分步版”，和 `run_pipeline` 执行逻辑一致。

In [ ]:
transactions = prepare_transactions(tables["transactions"])
article_department = prepare_article_department(tables["articles"])
customer_age_bin = prepare_customer_age_bin(tables["customers"])

ranker_config = load_ranker_from_cache(output_dir=OUTPUT_DIR)
print("Selected ranker:", describe_ranker(ranker_config))

submission = generate_submission(
    tables=tables,
    transactions=transactions,
    article_department=article_department,
    customer_age_bin=customer_age_bin,
    ranker_config=ranker_config,
    output_dir=OUTPUT_DIR,
)

submission.head(5)


## 4. 检查输出路径

In [ ]:
output_path = Path(OUTPUT_DIR) / "submission.csv"
print(output_path)
print("exists:", output_path.exists())

## 5. 生成层：一键运行版

如需要一条命令跑完整生成流程，可使用下面入口。

In [ ]:
state = run_pipeline(BASE_PATH)
state["submission"].head(5)

## 6. Evaluation Layer (Optional, Run Separately)

Evaluation and tuning are separated into `Final_Project_Eval.py` to keep generation fast.

Run in a separate session when needed:
```bash
python Final_Project_Eval.py
```

After evaluation, sync experiment records:
- `experiments/tuning_runs/<run_id>/`
- `experiments/runs_index.csv`
- after manual Kaggle submit, update `experiments/kaggle_scores.csv`


In [ ]:
from Final_Project_Eval import run_eval_pipeline

# 可选：这一步计算耗时较长，默认注释
# eval_state = run_eval_pipeline(BASE_PATH)
# eval_state["fold_metrics"]

## 8. 消融实验与可解释性分析

完成消融实验与 SHAP 分析后，可在 Kaggle 上打开并运行：

```text
LGBM_Ablation_SHAP_Analysis.ipynb
```

或直接运行脚本版：

```bash
pip install -q lightgbm shap
python LGBM_Ablation_SHAP_Analysis.py
```

输出产物：
- `outputs/lgbm_ablation_top5.csv` — 消融指标
- `outputs/shap_beeswarm.png` — SHAP bee swarm 图
- `outputs/shap_waterfall_sample0.png` — SHAP waterfall 图
- `outputs/shap_mean_abs_importance.csv` — 特征重要性排序

已完成的分析报告：
- [homework/HM_LGBM_Ablation_SHAP_Report.md](homework/HM_LGBM_Ablation_SHAP_Report.md)
- [homework/HM_LGBM_Ablation_SHAP_Report.pdf](homework/HM_LGBM_Ablation_SHAP_Report.pdf)

## 9. 实验追踪与管理

所有调参运行都统一记录在 `experiments/` 目录：

### 离线运行流程

1. **生成离线 run**（本地或 Kaggle）
   ```bash
   python Final_Project_Eval.py  # 评估 5 折 + 轻量调参
   ```
   
   或运行 LightGBM 版本（自动生成 submission）：
   ```python
   %env LGBM_TRAIN_CUSTOMER_CAP=80000
   # ... 其他参数设置 ...
   !python Final_Project_LGBM.py
   ```

2. **记录离线成绩**
   - 脚本自动生成 `outputs/offline_fold_metrics.csv` 和 `outputs/ranker_tuning_metrics.csv`
   - 手动创建 run 快照目录：`experiments/tuning_runs/<run_id>/`
   - 复制关键文件：
     - `offline_fold_metrics.csv` — 5 折评估指标
     - `ranker_tuning_metrics.csv` — 调参候选及平均分
     - `selected_ranker.csv` — 最优配置
     - `params.json` — 参数记录
     - `submission_sha256.txt` — submission 校验和

3. **更新 `experiments/runs_index.csv`**
   添加一行记录：`run_id | offline_mean_map12 | best_config | submission_sha256 | notes`

### 官方成绩同步

手动提交到 Kaggle 后，更新 `experiments/kaggle_scores.csv`：

| run_id | submission_variant | public_score | private_score | submitted_date |
|--------|-------------------|--------------|---------------|----------------|
| run_2026-04-27_001_hybrid_multisource | Final_Project_LGBM.py | 0.02679 | 0.02717 | 2026-04-29 |

### 台账一致性

- `README.md` 第 8 节"当前最佳离线记录"应与 `experiments/` 保持同步
- 每次新 run 都要追加到 `experiments/runs_index.csv` 和 `experiments/kaggle_scores.csv`

## 10. Kaggle 提交流程

### 步骤

1. **在 Kaggle Notebook 中运行 LightGBM**
   - 使用推荐参数配置（见第 7 节）
   - 生成 `submission.csv`

2. **下载并验证 submission**
   ```bash
   # 在 Kaggle 工作目录
   cp outputs/submission.csv /kaggle/working/submission.csv
   
   # 本地验证格式
   python -c "import polars as pl; df = pl.read_csv('submission.csv'); print(df.head()); print(f'Rows: {df.height}')"
   ```

3. **手动提交到 Kaggle**
   - 进入 Kaggle Competition 页面
   - 点击"Make Submission"
   - 选择文件并提交

4. **记录成绩**
   - 获取 Public Score 和 Private Score
   - 计算提交文件哈希：`sha256sum submission.csv`
   - 更新 `experiments/kaggle_scores.csv`

### 提交文件格式检查清单

- [ ] 文件名：`submission.csv`
- [ ] 列名：`customer_id,prediction`（无空格）
- [ ] 行数：与样本 submission 中的客户数一致
- [ ] 每行最多 12 个 article_id（空格分隔）
- [ ] 无重复 article_id（每行内）
- [ ] 客户 ID 完整覆盖
- [ ] 无前导零丢失（UTF-8 编码）

## 11. 数据路径配置

### 本地运行

推荐设置环境变量指向数据目录：

```bash
export HNM_DATA_PATH="/path/to/h-and-m-personalized-fashion-recommendations"
python Final_Project.py
```

或在脚本中修改 `BASE_PATH`。

支持的路径优先级（自动检测）：
1. 显式传入的路径
2. `HNM_DATA_PATH` 环境变量
3. `./data/h-and-m-personalized-fashion-recommendations`
4. `./h-and-m-personalized-fashion-recommendations`
5. `$PWD/data/h-and-m-personalized-fashion-recommendations`
6. `$PWD/h-and-m-personalized-fashion-recommendations`

### Kaggle 上运行

```python
import os
os.environ["HNM_DATA_PATH"] = "/kaggle/input/h-and-m-personalized-fashion-recommendations"

!python Final_Project.py
```

### 必需数据文件

- `articles.csv`
- `customers.csv`
- `transactions_train.csv`
- `sample_submission.csv`

## 12. 故障排查

### 常见问题

| 问题 | 原因 | 解决方案 |
|------|------|--------|
| `FileNotFoundError` | 数据路径错误 | 检查 `HNM_DATA_PATH` 或修改 `BASE_PATH` |
| `提交行数错误` | 未包含全部客户 | 确保 `submission.csv` 包含 sample_submission 中的所有客户 ID |
| `提交列名错误` | 列名拼写或空格错误 | 检查列名：`customer_id,prediction`（无多余空格） |
| `每行超过 12 个 article_id` | 预测列表未截断 | 检查 `MAX_K = 12` 设置 |
| `LGBM 内存溢出` | 候选数或训练集过大 | 降低 `LGBM_MAX_CANDIDATES_PER_CUSTOMER` 或 `LGBM_TRAIN_CUSTOMER_CAP` |
| `Kaggle 运行超时` | 参数配置过重 | 使用轻量配置或减少客户数限制 |

### 快速诊断

```python
import polars as pl

# 检查 submission 格式
submission = pl.read_csv("outputs/submission.csv")
print(f"Rows: {submission.height}")
print(f"Columns: {submission.columns}")
print(f"Max items per row: {submission.with_columns(pl.col('prediction').str.split(' ').list.len().alias('n_items'))['n_items'].max()}")
print(f"Null check: {submission.null_count()}")
```

## 13. 架构快速参考

### 核心文件职责

```
Final_Project.py
├─ 加载数据
├─ 构建推荐工件（用户历史、热门商品等）
├─ 对所有客户进行排序预测
└─ 输出 outputs/submission.csv

Final_Project_Eval.py（可选）
├─ 5 折交叉验证
├─ 规则融合调参
├─ 计算离线 MAP@12
└─ 保存最优权重到 ranker_tuning_metrics.csv

Final_Project_LGBM.py（可选，推荐）
├─ 生成多源候选池
├─ 特征工程与清洗
├─ LightGBM 二分类排序
├─ 时间窗口验证
└─ 输出 outputs/submission.csv

LGBM_Ablation_SHAP_Analysis.*（可选）
├─ 特征消融实验
├─ SHAP 可解释性分析
└─ 生成报告与可视化
```

### 推荐流程

```
1. 本地开发
   Final_Project.py  → submission.csv（快速验证）

2. 离线调参（可选）
   Final_Project_Eval.py  → 最优权重

3. Kaggle 冲分（推荐）
   Final_Project_LGBM.py  → 更强 submission.csv

4. 可解释性分析（可选）
   LGBM_Ablation_SHAP_Analysis.py  → 消融报告

5. 提交 Kaggle
   手动上传 submission.csv  → 获得 Public/Private 分数

6. 记录实验
   更新 experiments/ 台账  → 保留历史
```

### 评估指标

- **离线指标**：MAP@12（Mean Average Precision @ 12）
  - 对每个测试集客户，计算 AP@12
  - 取所有客户的平均值
  
- **Kaggle 指标**
  - **Public Score**：全量样本的前 50% 评估
  - **Private Score**：全量样本的后 50% 评估（最终排名依据）
  
- **消融指标**
  - 特征重要性排序
  - Gain / Split / Cover 分布
  - SHAP 平均绝对值

## 14. 进阶配置参考

### 环境变量完整列表（Final_Project_LGBM.py）

| 变量 | 默认值 | 说明 |
|------|--------|------|
| `LGBM_TRAIN_CUSTOMER_CAP` | 80000 | 训练客户数上限 |
| `LGBM_VALIDATION_CUSTOMER_CAP` | 60000 | 验证客户数上限 |
| `LGBM_MAX_CANDIDATES_PER_CUSTOMER` | 100 | 单客户最大候选数 |
| `LGBM_BASELINE_RECALL_TOP` | 80 | Baseline 召回 TOP-N |
| `LGBM_ATTRIBUTE_RECALL_TOP` | 12 | 属性召回 TOP-N |
| `LGBM_RECENT_GLOBAL_RECALL_TOP` | 18 | 最近全局召回 TOP-N |
| `LGBM_N_ESTIMATORS` | 450 | LightGBM 树的数量 |
| `LGBM_ENABLE_COOCCURRENCE_RECALL` | True | 启用共现召回 |
| `LGBM_COOCCURRENCE_RECALL_TOP` | 12 | 共现召回 TOP-N |
| `LGBM_ENABLE_COLD_START_RECALL` | True | 启用冷启动召回 |
| `LGBM_COLD_START_RECALL_TOP` | 18 | 冷启动召回 TOP-N |
| `LGBM_FEATURE_EXPERIMENT` | best | 特征集（base/best/all/add_*） |
| `LGBM_DROP_NOISY_FEATURES` | False | 去除噪声特征 |
| `LGBM_TRAIN_WINDOW_COUNT` | 1 | 训练时间窗口数（1 或 2） |

### 特征集选项（LGBM_FEATURE_EXPERIMENT）

- `best`：移除 `price_diff_user_item`（默认推荐）
- `base`：不使用新增特征
- `all`：使用所有特征
- `add_item_pop_ratio_7d_30d`：仅添加 7 天/30 天热度比
- `add_item_pop_ratio_30d_all`：仅添加 30 天/全期热度比
- `add_price_diff_user_item`：仅添加价格差异特征

### 典型场景配置

**场景 1：快速验证（本地/Kaggle）**
```python
%env LGBM_TRAIN_CUSTOMER_CAP=20000
%env LGBM_VALIDATION_CUSTOMER_CAP=15000
%env LGBM_MAX_CANDIDATES_PER_CUSTOMER=60
%env LGBM_N_ESTIMATORS=200
```

**场景 2：轻量冲分**
```python
%env LGBM_TRAIN_CUSTOMER_CAP=80000
%env LGBM_VALIDATION_CUSTOMER_CAP=60000
%env LGBM_MAX_CANDIDATES_PER_CUSTOMER=100
%env LGBM_BASELINE_RECALL_TOP=80
%env LGBM_N_ESTIMATORS=450
```

**场景 3：重型冲分（需要 GPU/更长运行时间）**
```python
%env LGBM_TRAIN_CUSTOMER_CAP=120000
%env LGBM_VALIDATION_CUSTOMER_CAP=80000
%env LGBM_MAX_CANDIDATES_PER_CUSTOMER=150
%env LGBM_BASELINE_RECALL_TOP=100
%env LGBM_TRAIN_WINDOW_COUNT=2
%env LGBM_N_ESTIMATORS=600
```

## 15. 相关文档导航

### 项目文档

- [README.md](README.md) — 项目总体介绍与运行方式
- [task.md](task.md) — 任务书要求映射与完成状态
- [AI_Assistant_Memo.md](AI_Assistant_Memo.md) — 工作规范与流程

### 实验与分析

- [experiments/README.md](experiments/README.md) — 实验追踪目录说明
- [experiments/runs_index.csv](experiments/runs_index.csv) — 离线运行总索引
- [experiments/kaggle_scores.csv](experiments/kaggle_scores.csv) — Kaggle 分数记录
- [homework/HM_LGBM_Ablation_SHAP_Report.md](homework/HM_LGBM_Ablation_SHAP_Report.md) — 消融与 SHAP 报告

### 脚本文件

- `Final_Project.py` — 生成层（推荐开始阅读）
- `Final_Project_Eval.py` — 评估层与调参
- `Final_Project_LGBM.py` — 增强版 LightGBM 流水线
- `LGBM_Ablation_SHAP_Analysis.py` — 可解释性分析脚本

### 数据来源

- Kaggle Dataset: [H&M Personalized Fashion Recommendations](https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations)

In [ ]:
# 可选：验证输出文件
from pathlib import Path
import polars as pl

submission_path = Path(OUTPUT_DIR) / "submission.csv"

if submission_path.exists():
    print(f"✓ Submission file found: {submission_path}")
    
    submission = pl.read_csv(submission_path)
    print(f"\n数据行数: {submission.height}")
    print(f"列名: {submission.columns}")
    
    # 验证格式
    token_stats = submission.with_columns(
        pl.col("prediction").str.split(" ").list.len().alias("n_items")
    )
    print(f"\n每行推荐数范围: {token_stats['n_items'].min()} - {token_stats['n_items'].max()}")
    print(f"超过 12 个的行数: {token_stats.filter(pl.col('n_items') > 12).height}")
    print(f"为空的行数: {token_stats.filter(pl.col('n_items') == 0).height}")
    
    print("\n样本预览:")
    print(submission.head(3))
else:
    print(f"✗ Submission file not found: {submission_path}")
    print(f"  Please run generation step first.")

## 7. LightGBM 增强版（可选，在 Kaggle 运行）

LightGBM 增强版实现了候选扩展、特征工程和排序模型，通常在 Kaggle Notebook 中运行。

关键特性：
- **候选扩展**：baseline + 属性召回 + 共现召回 + 冷启动召回
- **特征工程**：用户行为、商品热度、消融特征等
- **排序模型**：二分类 LightGBM 模型
- **环境变量控制**：灵活调整模型参数

在 Kaggle Notebook 中运行（推荐配置）：

```python
%env LGBM_TRAIN_CUSTOMER_CAP=80000
%env LGBM_VALIDATION_CUSTOMER_CAP=60000
%env LGBM_MAX_CANDIDATES_PER_CUSTOMER=100
%env LGBM_BASELINE_RECALL_TOP=150
%env LGBM_ATTRIBUTE_RECALL_TOP=8
%env LGBM_RECENT_GLOBAL_RECALL_TOP=12
%env LGBM_ENABLE_COLOUR_RECALL=0
%env LGBM_ENABLE_SECTION_RECALL=0
%env LGBM_ENABLE_COOCCURRENCE_RECALL=1
%env LGBM_COOCCURRENCE_RECALL_TOP=12
%env LGBM_TRAIN_WINDOW_COUNT=1
%env LGBM_FEATURE_EXPERIMENT=best
%env LGBM_DROP_NOISY_FEATURES=0
%env LGBM_N_ESTIMATORS=450

!python Final_Project_LGBM.py
!cp outputs/submission.csv /kaggle/working/submission.csv
```

更多配置说明详见 [README.md](README.md) 第 5 节。

## 16. 轻度 Optuna 超参优化（可选，预期 +1~2%）

### 背景

虽然 LightGBM baseline 已达到 Public 0.02679 / Private 0.02717，但核心超参（learning_rate、num_leaves、max_depth 等）仍有优化空间。Optuna 可在**有限搜索空间**（20-25 trials）下找到更优的超参组合，预期可追加 +1~2% 的离线提升。

### 快速启动

**前置条件**：
- 时间充足（离截止 > 10 天）
- Kaggle 提交次数充足（> 20 次）
- 现有 LightGBM baseline 分数已稳定（Private Score 变动 < 0.001）

**步骤**：

1. **安装 Optuna**
   ```bash
   pip install optuna>=3.1.0
   ```

2. **创建搜索脚本** `Final_Project_Optuna.py` 或在 Kaggle Notebook 中运行：
   ```python
   import optuna
   from optuna.samplers import TPESampler
   
   def objective(trial):
       # 搜索空间定义
       params = {
           'learning_rate': trial.suggest_float('lr', 0.01, 0.2, log=True),
           'num_leaves': trial.suggest_int('leaves', 20, 100),
           'max_depth': trial.suggest_int('depth', 5, 15),
           'feature_fraction': trial.suggest_float('f_frac', 0.6, 1.0),
           'bagging_fraction': trial.suggest_float('b_frac', 0.6, 1.0),
           'lambda_l1': trial.suggest_float('l1', 0, 5),
           'lambda_l2': trial.suggest_float('l2', 0, 5),
       }
       
       # 使用这些参数训练 LightGBM
       map12_score = train_lgbm_with_params(params)
       return map12_score
   
   # 轻量搜索：仅 20 trials
   sampler = TPESampler(seed=610)
   study = optuna.create_study(direction='maximize', sampler=sampler)
   study.optimize(objective, n_trials=20, n_jobs=1)
   
   # 保存结果
   best_params = study.best_params
   best_value = study.best_value
   print(f"Best MAP@12: {best_value}")
   print(f"Best params: {best_params}")
   ```

3. **结果保存**
   ```bash
   outputs/optuna_best_params.json
   outputs/optuna_study_history.csv
   outputs/optuna_best_validation_map12.txt
   ```

4. **对比验证**
   - 用最优超参重新训练 LightGBM
   - 生成新 submission.csv
   - 在 Kaggle 提交并记录 Public/Private 分数
   - 更新 `experiments/runs_index.csv`

### 预期收益与成本

| 指标 | 预期值 |
|------|--------|
| 离线提升 | +0.5~1.5% （MAP@12 从 0.0306 → 0.0311~0.0313） |
| 线上提升（预期）| +1~3% （取决于泛化性） |
| 搜索时间 | 20 trials × 2 min = 40 分钟 |
| 开发成本 | 3-5 小时（脚本集成 + 验证） |
| 风险 | 有限搜索可能陷入局部最优 |

### 保守方案

如果时间紧张，可跳过 Optuna，改用**轻量网格搜索**（5-8 组关键参数）：

```python
# 仅尝试关键参数的几组组合
for lr in [0.05, 0.08, 0.1, 0.15]:
    for leaves in [31, 50, 70]:
        # 训练 & 评估
        # 成本：15-25 分钟，不需要 Optuna
```